Notes: 

cell builder is currently embedded in the simulation module... Need to build the 

Also want to implement new clustering method...

In [1]:
import sys
sys.path.append('..')
sys.path.append('../Modules')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(os.getcwd())
if 'notebook' in os.getcwd():
    # os.chdir("../scripts") # go from Neural-Modeling/notebooks to Neural-Modeling/scripts, where simulation outputs will be generated to. (maybe a separate folder could be used...)
    os.chdir("../simulations") # go to output folder
    print(os.getcwd())

/home/drfrbc/Neural-Modeling/notebooks
/home/drfrbc/Neural-Modeling/simulations


User Specifications -  get parameters, simulation folder

In [2]:
from Modules.constants import HayParameters
import datetime
import pickle
from neuron import h

sim_set_title = "description_of_simulation_set"
# create simulation folder
sims_dir = f"{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M')}-{sim_set_title}"
os.makedirs(sims_dir, exist_ok=True)

# this is a quick example, but the generation of parameter sets and titles is in scripts/gen_param_list_advanced.py
# @TODO: implement generating these as seen in gen_param_list_advanced.py
sim_titles = ["description_of_simulation"]
parameter_sets = [HayParameters(sim_title) for sim_title in sim_titles] # example parameter set

# @TODO: compile modfiles once

# create simulation folders and save parameters in them
for parameters, sim_title in zip(parameter_sets, sim_titles):
    # create simulation folder
    sim_dir = os.path.join(sims_dir, sim_title)
    os.makedirs(sim_dir, exist_ok=True)

    with open(os.path.join(sim_dir, "params.pickle"), 'wb') as file:
        pickle.dump(parameters, file)

    # load modfiles
    try:
        h.load_file('stdrun.hoc')
        # h.nrn_load_dll('./x86_64/.libs/libnrnmech.so' # IF IN SCRIPTS FOLDER
        load_modfiles = h.nrn_load_dll('../scripts/x86_64/.libs/libnrnmech.so') # IF IN SIMULATIONS FOLDER
        if load_modfiles != 1:
            raise Exception("Error loading mod files")
        else:
            print("Mod files loaded successfully")
    except:
        # Already loaded
        pass 

# @TODO: at simulation time, the parameters should be loaded from each sim_dir in sims_dir. 
# We can make one script that does this code snippet, then passes sims_dir to the mpiexec simulation script.

Mod files loaded successfully


--No graphics will be displayed.


Build cell to save segments csv

In [ ]:
from Modules.cell_builder import CellBuilder
from Modules.logger import Logger
from Modules.cell_builder import SkeletonCell
# this script would be called with mpiexec

# @TODO: would we want to use mpiexec for this script? It depends on if there are different morphologies: different cell types, different reduction parameters, and segmentation. Yes, one per simulation folder. might as well.
# Additionally, do we save these in simulation folders or try to use a common folder? A: simulation folder

# for rank in [0]: # replace with actual mpi implementation


def generate_segments_csv(sim_dir, parameters, logger): #@TODO: add this to some class... Maybe not CellBuilder because the CellBuilder instance could be temporary instead? discuss with @davidfague
    # build the cell
    logger.log(f"Building cell to generate segments.csv")
    cell_builder = CellBuilder(getattr(SkeletonCell, parameters.skeleton_cell_type), parameters, logger)
    cell, _ = cell_builder.build_cell()
    logger.log(f"Cell built successfully.")

    logger.log(f"Changing cell morphology, segmentation, etc")
    # manipulate morphology: reduction, segmentation
    #@TODO add code from cellbuilder.py: CellBuilder.build_cell  -lines around reductor code block
    ####
    ####
    logger.log(f"Finished changing cell morphology, segmentation, etc")

    logger.log("Saving adjacency matrix")
    if parameters.save_adj_matrix:
        adj_matrix = cell.compute_directed_adjacency_matrix()
        np.savetxt(os.path.join(parameters.path, "adj_matrix.txt"), adj_matrix)
    logger.log("Finished saving adjacency matrix")

    logger.log("Getting segments data")

    # save segments csv in simulation folder - the rest of this cell
    #@TODO: Make modularized code for this and clean. standardize between here and cell_model (this is from simulation.py)
    #@TODO: clean up cell.get_segments alongside cell.get_segments_of_type
    #@TODO: add sec_type (or another name for the variable) for denoting the segment type at the 'distal_basal' level instead of 'dend' for example.
    # Classify segments by morphology, save coordinates
    segments, seg_data = cell.get_segments(["all"]) # (segments is returned here to preserve NEURON references)
    seg_sections = []
    seg_idx = []
    seg_coords = []
    seg_half_seg_RAs = []
    seg = []
    seg_Ls = []
    sec_Ls = []
    sec_Ds = []
    seg_distance = []
    psegs=[]
    
    for i,entry in enumerate(seg_data):
        # if parameters.build_stylized: #@DEPRACATED
        #     sec_name = entry.section.split(".")[-1]
        # else:
        sec_name = entry.section.split(".")[-1] # name[idx]
        #print(f"sec_name: {sec_name}")
        seg_sections.append(sec_name.split("[")[0])
        seg_idx.append(sec_name.split("[")[1].split("]")[0])
        seg_coords.append(entry.coords)
        seg_half_seg_RAs.append(entry.seg_half_seg_RA)
        seg.append(entry.seg)
        seg_Ls.append(entry.L)
        psegs.append(entry.pseg)
        sec_Ls.append(segments[i].sec.L)
        sec_Ds.append(segments[i].sec.diam)
        seg_distance.append(h.distance(segments[0], segments[i]))
        
        
    seg_sections = pd.DataFrame({ #@TODO: rename seg_sections to seg_sec_data or something
        "section": seg_sections, 
        "idx_in_section_type": seg_idx,
        "seg_half_seg_RA": seg_half_seg_RAs,
        "L": seg_Ls,
        "seg":seg,
        "pseg":psegs,
        "Section_L":sec_Ls,
        "Section_diam":sec_Ds,
        "Distance":seg_distance,
        })

    seg_coords = pd.concat(seg_coords)

    seg_data = pd.concat((seg_sections.reset_index(drop = True), seg_coords.reset_index(drop = True)), axis = 1) #@TODO: compute these together instead or make seg_sections computation more concise?
    seg_data = seg_data.reset_index(drop=True) #@TODO: check if this is needed
    seg_data['seg_id'] = seg_data.index # add a seg_id so that row i → seg_id i
    seg_data.to_csv(os.path.join(sim_dir, "segment_data1.csv"))
    logger.log("Saved segments data to segment_data.csv")

    ### new version @TODO: finish this implementation for adding the new sec_types to segments.csv
    sec_types_to_get = np.unique([sec_type for syn_properties in [parameters.exc_syn_properties, parameters.inh_syn_properties] for sec_type in syn_properties.keys()])
    print(f"getting segments of types: {sec_types_to_get} for synapses")
    # These are the types the method handles currently:
    # sec_types_to_get = [
    #     'soma',
    #     'perisomatic',
    #     'trunk',
    #     'distal_basal',
    #     'distal_apic',
    #     'nexus',
    #     'tuft',
    #     'oblique'
    # ]
    rows = []
    for stype in sec_types_to_get:
        try:
            segs = cell.get_segments_of_type(stype)
        except ValueError:
            # in case a type is empty / not implemented
            continue
        for seg in segs:
            rows.append({
                'sec_name': seg.sec.name(),  # e.g. "/cell/apic[12]"
                'seg_x':    seg.x,           # normalized position along the section
                'sec_type': stype,
                'seg_id': segments.index(seg), # index of the segment in the list returned from cell.get_segments(['all'])
            })

    df = pd.DataFrame(rows, # @TODO: check this dataframe. remove duplicate segments. include segment id from the index of the list returned from cell.get_segments(['all'])
            columns=['sec_name','seg_x','sec_type'])
    df.to_csv(os.path.join(sim_dir, "segment_data2.csv"), index=False)
    ###

    ### combining main seg_data with this precise sec_type
    # print out any seg_ids that have more than one precise type #@TODO: debugging overlapping precise sec_types that may need clearer definitions or stricter logic
    # (e.g. "perisomatic" and "trunk")
    # group to collect all sec_type per seg_id
    grouped = df.groupby('seg_id')['sec_type'].unique()
    overlaps = grouped[grouped.apply(lambda arr: len(arr) > 1)]
    if not overlaps.empty:
        print("Segments with multiple precise sec_types:")
        for sid, types in overlaps.items():
            print(f"  seg_id {sid}: {types.tolist()}")

    # 5) build a map: if there's exactly one type, keep it; otherwise None
    precise_map = grouped.apply(lambda arr: arr[0] if len(arr) == 1 else None)

    # 6) assign into your main DataFrame
    seg_data['sec_type_precise'] = seg_data['seg_id'].map(precise_map)

    # 7) write the integrated CSV back out (overwriting the old one)
    seg_data.to_csv(os.path.join(sim_dir, "segment_data.csv"), index=False)
    logger.log("Saved integrated segment_data.csv with sec_type_precise")


from collections.abc import Callable, Iterable
from typing import Mapping, Union

def run_on_all_sims( #TODO: Use MPI to process these in parallel. Evenly Distribute sims to workers instead of using 1 per rank. add to class?
    sims_dir: str,
    process_fns: Union[Callable[[str, Mapping, object], None],
                       Iterable[Callable[[str, Mapping, object], None]]]
) -> None:
    """process_fns can be a single function or a list of functions. 
    Each function should take the simulation directory, parameters, and logger as arguments."""
    # normalize to a list
    if callable(process_fns):
        fns = [process_fns]
    else:
        fns = list(process_fns)

    for entry in os.listdir(sims_dir):
        sim_dir = os.path.join(sims_dir, entry)
        if not os.path.isdir(sim_dir):
            continue

        # load parameters
        with open(os.path.join(sim_dir, "params.pickle"), "rb") as f:
            parameters = pickle.load(f)

        logger = Logger(sim_dir) # create per‑sim logger (write info into "sims_dir/sim_dir/log.txt")

        for fn in fns :# run each processing function
            fn(sim_dir=sim_dir, parameters=parameters, logger=logger)

# for sim_dir in os.listdir(sims_dir):
#     sim_dir = os.path.join(sims_dir, sim_dir)
#     if not os.path.isdir(sim_dir):
#         continue

#     # load parameters
#     with open(os.path.join(sim_dir, "params.pickle"), 'rb') as file:
#         parameters = pickle.load(file)

#     logger = Logger(sim_dir)

#     generate_segments_csv(sim_dir, parameters)

run_on_all_sims(sims_dir, generate_segments_csv) # generate the segments csv for each simulation

Removing duplicate coordinate at index 1 in section L5PCtemplate[3].apic[0]


In [7]:
ls

2025-04-16-14-39-Test/
2025-04-16-14-44-Test/
2025-04-16-16-35-description_of_simulation_set/
2025-04-17-14-17-description_of_simulation_set/
2025-04-17-14-18-description_of_simulation_set/
2025-04-17-14-23-description_of_simulation_set/
2025-04-17-14-26-description_of_simulation_set/
2025-04-17-14-27-description_of_simulation_set/
2025-04-17-14-28-description_of_simulation_set/
2025-04-17-14-31-description_of_simulation_set/
2025-04-17-14-32-description_of_simulation_set/
2025-04-17-14-39-description_of_simulation_set/
2025-04-17-15-00-description_of_simulation_set/
builder_runtime.txt
replace_runtime.txt


In [9]:
os.listdir(sim_dir)

['params.pickle', 'log.txt', 'runtimes.csv', 'segment_data.csv']

In [11]:
segments = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))

In [ ]:
segments2 = pd.read_csv(os.path.join(sim_dir, "segment_data2.csv"))

In [13]:
segments.columns

Index(['Unnamed: 0', 'section', 'idx_in_section_type', 'seg_half_seg_RA', 'L',
       'seg', 'pseg', 'Section_L', 'Section_diam', 'Distance', 'p0_0', 'p0_1',
       'p0_2', 'pc_0', 'pc_1', 'pc_2', 'p1_0', 'p1_1', 'p1_2', 'r', 'dl_0',
       'dl_1', 'dl_2'],
      dtype='object')

Generate synapse Locations - load segments csv

In [ ]:
# load segments csv

## generate synapse locations - code from cellbuilder @TODO: need to change to utilize segment_data.csv instead of cell_model.py

# save to synapses csv in simulation folder # example in cell_builder of getting properties

import pandas as pd
from functools import partial
# define a class for generating synapses abstractly
class PreSimSynapseGenerator:
    def __init__(self, segments, parameters, logger):
        self.segments = segments
        self.parameters = parameters
        self.synapses = pd.DataFrame()
        self.logger = logger

        if self.parameters.segment_measurement_for_probabilities not in ['length', 'surface_area']:
            raise ValueError(f"Measurement for probabilities must be 'length' or 'surface_area'. Not {self.parameters.segment_measurement_for_probabilities}.")

    def generate_synapse_locations(self):
        for syn_properties_set in [self.parameters.exc_syn_properties, self.parameters.inh_syn_properties]:
            for sec_type, synapse_properties in syn_properties_set.items():
                self.logger.log(f"Generating synapses for {sec_type.upper()} with properties: {synapse_properties}")
                segments_to_generate_on = self.get_segments_of_type(sec_type, self.segments) # get segments
                self.random_state = np.random.RandomState(self.parameters.inh_syn_properties[sec_type]['seed']['synapses'])
                np.random.seed(self.parameters.inh_syn_properties[sec_type]['seed']['synapses'])

                synapses_this_sec_type = self.build_synapses_with_specs(segments_to_generate_on = segments_to_generate_on,
                                        sec_type = sec_type,
                                        synapse_type = synapse_properties['synapse_type'],
                                        use_density = self.parameters.use_density,
                                        syn_number = synapse_properties['syn_number'] if not self.parameters.use_density else None,
                                        syn_density = synapse_properties['syn_density'] if self.parameters.use_density else None,
                                        initial_weight_distribution = synapse_properties['initial_weight_distribution'],
                                        release_probability_distribution = synapse_properties['release_probability_distribution'],
                                        name = f"inh_{sec_type}"
                                        )
                self.synapses = pd.concat((self.synapses, synapses_this_sec_type), ignore_index=True)
        self.synapses = self.synapses.reset_index(drop=True) #@TODO: check if this is necessary and check self.synapses.

    def get_segments_of_type(self, sec_type, segments):
        return segments[segments['sec_type'] == sec_type]
    
    def build_synapses_with_specs(self, segments_to_generate_on: pd.DataFrame, sec_type: str, synapse_type: str, use_density: bool,
                                  syn_number: int, syn_density: float, initial_weight_distribution: dict,
                                  release_probability_distribution: dict, name: str, syn_mod:str, syn_params_choices) -> pd.DataFrame:
        
        if initial_weight_distribution is None or release_probability_distribution is None:
            raise ValueError("Both gmax_dist_params and P_release_params must be provided.")
        
        if syn_number is None and syn_density is None:
            raise ValueError("Either syn_number or syn_density must be provided.")
        elif syn_number is not None and syn_density is not None:
            raise ValueError("Only one of syn_number or syn_density must be provided.")
        elif syn_number is not None and use_density:
            raise ValueError("syn_number should be none when using density.")
        elif syn_density is not None and not use_density:
            raise ValueError("syn_density should be none when not using density.")

        #@TODO: check that this doesn't throw an error with expected inputs.
        #@TODO: move initial_weight_distribution info to a dictionary within synapse_properties, same for release probability_distribution.
        if not isinstance(syn_params_choices, list):
            raise ValueError(f"syn_params_choices must be a list. Not {type(syn_params_choices)}. syn_params_choices: {syn_params_choices}.")
        if False in [True if not isinstance(syn_params, dict) else False for syn_params in syn_params_choices]:
            raise ValueError(f"syn_params_choices must be a list of dictionaries. Not {type(syn_params_choices)}. syn_params_choices: {syn_params_choices}.")

        if initial_weight_distribution['function'] is not None:
            initial_weight_distribution = partial(initial_weight_distribution['function'], **initial_weight_distribution['params'], size=1)
            initial_weight_distribution_is_partial = True
        else:
            initial_weight_distribution = initial_weight_distribution['params']['mean'] # use mean if no function is provided
            initial_weight_distribution_is_partial = False
        self.logger.log(f"Initial weight distribution for {name}: {initial_weight_distribution}")

        if release_probability_distribution['function'] is not None: #@TODO: update parameters so that release_probability_distribution is defined in syn_properties
            release_probability_distribution = partial(release_probability_distribution['function'], **release_probability_distribution['params'], size=1)
            release_probability_distribution_is_partial = True
        else:
            release_probability_distribution = release_probability_distribution['params']['mean']
            release_probability_distribution_is_partial = False
        self.logger.log(f"Release probability distribution for {name}: {release_probability_distribution}")

        # calculate probabilities of placing synapses on segments
        total_measurement = segments_to_generate_on[self.parameters.segment_measurement_for_probabilities].sum()
        self.logger.log(f"Total {self.parameters.segment_measurement_for_probabilities} for {name}: {total_measurement}")
        segments_to_generate_on['probability'] = segments_to_generate_on[self.parameters.segment_measurement_for_probabilities] / total_measurement

        if segments_to_generate_on['probability'].sum() != 1:
            raise ValueError("Probabilities do not sum to 1. Check your segment measurement for probabilities.")

        #@TODO: make sure that within 50 microns is excluded from non-perisomatic types when generating segments.csv

        if use_density:
            # calculate number of synapses per segment
            syn_number = total_measurement * syn_density #@TODO make compatible with syn_density being function instead of float (not at all urgent)
        self.logger.log(f"Number of synapses being generated for {name}: {syn_number}")

        synapses = pd.DataFrame(columns=['name', 'modfile', 'initW', 'gmax', 'release_probability', 'seg_id']) #@TODO: add columns for possible syn_params keys

        for _ in range(syn_number): # @TODO: do this in parallel instead of serial. Use list comprehension?
            # sample a segment
            segment = self.random_state.choice(segments_to_generate_on, 1, True, p=segments_to_generate_on['probability'])[0] #@TODO: check if [0] is necessary

            # choose sub-synapse type (short term plasticity, gbar, etc. properties)
            if 'exc' in name or 'AMPA' in syn_mod or 'pyr2pyr' in syn_mod: # choose between CS or CP #@TODO: check that syn_params_choices is expected type. tuple?
                syn_params_this_syn = self.random_state.choice(syn_params_choices, p=(0.9, 0.1)) # choose between CS2CP and CP2CP if it is AMPA. pyr2pyr will not be a tuple or list.
            elif 'inh' in name or 'GABA' in syn_mod or 'int2pyr' in syn_mod:
                syn_params_this_syn = syn_params_choices[1] if segment['Distance'] > 100 else syn_params_choices[0] # choosing between PV and SST

            # if not isinstance(syn_params_this_syn, dict): #@TODO: I moved this check to before loop instead of inside the loop, but need to check that it is the same
            #     raise ValueError(f"syn_params_this_syn must be a dictionary. Not {type(syn_params_this_syn)}. syn_params_this_syn: {syn_params_this_syn}.")

            # sample a release probability
            if release_probability_distribution_is_partial:
                syn_params_this_syn["release_probability"] = release_probability_distribution(size=1) # sample distribution
            else:
                syn_params_this_syn["release_probability"] = release_probability_distribution # distribution is a constant

            if 'int2pyr' in syn_mod or 'pyr2pyr' in syn_mod:  # these modfiles do release probability computation as spikes arrive during simulation instead of before
                syn_params_this_syn["P_0"] = syn_params_this_syn["release_probability"]
            else: # syn_mod does not have attribute for release probability so we approximate it by testing 1 release for entire simulation.
                p_test = self.random_state.uniform(low=0, high=1, size=1)
                if p_test < syn_params_this_syn["release_probability"]:
                    syn_params_this_syn["P_0"] = 1 #@TODO: when actually building synapses if syn_mod is not int2pyr or pyr2pyr then do not generate synapses with P_0 = 0. And skip assigning P_0 to the synapse object.
                else:
                    syn_params_this_syn["P_0"] = 0 #synapse is not releasing

            # sample an initial weight
            if initial_weight_distribution_is_partial:
                syn_params_this_syn["initW"] = initial_weight_distribution(size=1) # sample distribution
            else:
                syn_params_this_syn["initW"] = initial_weight_distribution # distribution is a constant

            # add row to dataframe
            synapses = pd.concat((synapses, pd.DataFrame({
                'name': f"{name}_{_}",
                'modfile': syn_mod,
                'initW': syn_params_this_syn["initW"],
                'gmax': syn_params_this_syn["gmax"],
                'release_probability': syn_params_this_syn["release_probability"],
                'seg_id': segment['seg_id'],
            }, index=[0])), ignore_index=True)

def use_pssg(sim_dir: str, parameters, logger):
    # load parameters
    with open(os.path.join(sim_dir, "params.pickle"), 'rb') as file:
        parameters = pickle.load(file)

    # load segments
    segments = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))

    # create PreSimSynapseGenerator instance
    pssg = PreSimSynapseGenerator(segments=segments, parameters=parameters, logger=logger)
    pssg.generate_synapse_locations()
    pssg.synapses.to_csv(os.path.join(sim_dir, "synapses.csv"), index=False)
    # return pssg 


run_on_all_sims(sims_dir, use_pssg)

KeyError: 'sec_type'

Generate synapse weights

In [ ]:
# load synapse locations from synapses csv
synapses = pd.read_csv(os.path.join(sim_dir, "synapses.csv"))

# generate synapse weights, params, etc based on locations

# save to synapses csv in simulation folder

#@V-Marco TODO: Should we generate weights in the same loop as synapse generation, or have a separate section for it?
# is it faster to compute if we do not repeat the loop? 
# If weights are generated separately, then we have some flexibility with what synapses.csv we pass, and making hand alterations to locations before generated weights. 
# If we made a hand alteration later then we might also need to intelligently alter the weight.
# weights are currently generated in PreSimSynapseGenerator.generate_synapse_locations.

Generate Spike Trains

In [ ]:
# USE cell_builder.assign_spikes and presynaptic.py for reference code. 
# @TODO: adapt presynaptic.py to use segments.csv instead of cell object.

# generate functional groups from params

# generate presynaptic cells from functional groups and params

# generate spike trains for presynaptic cells from params

# load synapse locations from synapses csv

# cluster synapse locations into presynaptic cells

# assign synapses to presynaptic cells

# save spike trains and synapse assignments as spike_trains.csv to simulation folder

Read synapse specifications and build cell with synapses

In [ ]:
# build synapses from synapses csv from simulation folder

# set spike trains from spike_trains.csv from simulation folder

